# Model info, cache status and loading progress

[Back to the introduction](demo.ipynb). Run in order. `LLM.model_info()` lists the files a model needs, their sizes and whether the Transformers.js browser cache already holds them, without downloading weights. `progress_callback` reports loading events to Python.

`download_bytes` is the download size, not runtime memory. `cache_status` only reflects the Transformers.js Cache API: hosted JupyterLite files and ZIPs in `/drive/models` are not counted. Unknown values are `None` or `"unknown"`, never guessed. An explicit `dtype` lets the check work offline.

In [ ]:
from lmlite import LLM, cached_models, clear_cache

model_id = "Xenova/distilgpt2"
dtype = "q8"


def show(info):
    total = "unknown" if info.download_bytes is None else f"{info.download_bytes / 1e6:.1f} MB"
    print(f"{info.model_id} ({info.task}, {info.backend}, {info.dtype}): cache={info.cache_status}, download={total}")
    for file in info.files:
        size = "?" if file["size"] is None else f"{file['size'] / 1e6:.2f} MB"
        print(f"  {'cached' if file['cached'] else 'remote':6} {size:>9}  {file['name']}")
    if info.error:
        print("  error:", info.error)

In [ ]:
show(await LLM.model_info(model_id, backend="wasm", dtype=dtype))

In [ ]:
def on_progress(event):
    # Print file completions and status changes, not every byte update.
    if event["status"] != "progress" or (event["total"] and event["loaded"] == event["total"]):
        print(event)


llm = await LLM.create(generator_model=model_id, backend="wasm", dtype=dtype, progress_callback=on_progress)

In [ ]:
show(await LLM.model_info(model_id, backend="wasm", dtype=dtype))
await llm.dispose()

## Cache manager

One line per model in the Transformers.js browser cache: architecture (from the cached `config.json`), task (from the Hub, `?` offline), name and size, with a button to delete it (all revisions). Enter a model ID and click Load to download it into the cache; it is loaded as a text-generation model and released right away. Hosted model files and ZIPs under `/drive/models` are not listed.

In [ ]:
import asyncio

import ipywidgets as widgets
from IPython.display import display


def columns(architecture, task, model, *buttons, style=None):
    # The header and model lines share column widths so they line up.
    cells = [(architecture, {"width": "8em"}), (task, {"width": "13em"}), (model, {"flex": "1"})]
    labels = [widgets.Label(text, style=style or {}, layout=widgets.Layout(**layout)) for text, layout in cells]
    return widgets.HBox([*labels, *buttons], layout=widgets.Layout(width="100%"))


class CacheManager:
    def __init__(self):
        self.task = None
        self.header = columns("Architecture", "Task", "Model", style={"font_weight": "bold"})
        self.rows = widgets.VBox()
        self.model = widgets.Text(placeholder="Xenova/distilgpt2", layout=widgets.Layout(flex="1"))
        self.load = widgets.Button(description="Load", button_style="primary")
        self.status = widgets.Label()
        self.view = widgets.VBox([self.header, self.rows, widgets.HBox([self.model, self.load]), self.status])
        self.load.on_click(lambda _: self.start(self.download))

    def set_busy(self, busy):
        for control in (self.load, *(row.children[-1] for row in self.rows.children)):
            control.disabled = busy

    def start(self, action):
        if self.task is None or self.task.done():
            self.set_busy(True)
            self.task = asyncio.create_task(self.refresh(action))

    async def refresh(self, action=None):
        try:
            if action:
                await action()
            self.show(await cached_models())
        except Exception as error:
            # Surface callback failures in the widget, not a background console.
            self.status.value = f"{type(error).__name__}: {error}"
        finally:
            self.set_busy(False)

    async def download(self):
        name = self.model.value.strip()
        if not name:
            raise ValueError("Enter a model ID first.")
        # shortcut: loads as text generation; embedding/classification models need LLM.create(embedding_model=...) etc.
        llm = await LLM.create(generator_model=name, progress_callback=self.on_progress)
        await llm.dispose()  # Only fill the cache; later loads read from it.

    def on_progress(self, event):
        self.status.value = f"{event['status']} {event['model']} {event['file']}".strip()

    async def delete(self, model):
        self.status.value = f"Deleting {model}..."
        await clear_cache(model=model)

    def show(self, models):
        for row in self.rows.children:
            row.close()
        self.rows.children = ()
        if models is None:
            self.status.value = "Cache inspection is unavailable in this browser."
            return
        # Non-model entries (the shared ONNX Runtime files) are not listed.
        models = [entry for entry in models if entry["model_id"]]
        rows = []
        for entry in models:
            model = entry["model_id"]
            size = "size unknown" if entry["size"] is None else f"{entry['size'] / 1e6:.1f} MB"
            delete = widgets.Button(description="Delete", button_style="danger")
            delete.on_click(lambda _, model=model: self.start(lambda: self.delete(model)))
            name = f"{model}@{entry['revision']} ({len(entry['files'])} files, {size})"
            rows.append(columns(entry["model_type"] or "?", entry["task"] or "?", name, delete))
        self.rows.children = rows
        known = sum(entry["size"] or 0 for entry in models)
        unknown = " (some sizes unknown)" if any(entry["size"] is None for entry in models) else ""
        self.status.value = f"{len(models)} cached, {known / 1e6:.1f} MB{unknown}" if models else "No models cached."


manager = CacheManager()
display(manager.view)
await manager.refresh()

Disposal keeps cached files. Clearing the cache removes every Transformers.js download, so the next load downloads again.

In [ ]:
print(await clear_cache())
show(await LLM.model_info(model_id, backend="wasm", dtype=dtype))